# WR Split-Half Stability

A leaderboard is only useful if it measures something that persists. This notebook asks whether wide receivers who create more separation than expected early in the sample also tend to do so later.

We use the WR-specific route-level predictions and compare:

```text
weeks 1-4 mean SOE
vs.
weeks 5-8 mean SOE
```

The goal is not a perfect correlation. Route running is noisy, opponent-dependent, and role-dependent. The question is whether the metric contains repeatable signal.

In [1]:
from pathlib import Path
import statistics
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from separation_over_expected.reports import pearson_correlation, read_csv_rows, split_half_stability_rows

PREDICTIONS = PROJECT_ROOT / "data" / "processed" / "position_baselines" / "route_level_baseline_predictions_wr.csv"
STABILITY = PROJECT_ROOT / "data" / "processed" / "stability" / "wr_split_half_stability.csv"

PROJECT_ROOT

PosixPath('/Users/moggo/Projects/NFL/Separation_Over_Expected')

## Build the Stability Table

Each row is a WR with at least 20 routes in weeks 1-4 and at least 20 routes in weeks 5-8.

In [2]:
routes = read_csv_rows(PREDICTIONS)
stability = split_half_stability_rows(routes, position="WR", min_routes_per_half=20)
len(stability)

107

In [3]:
stability[:15]

[{'nflId': '53478', 'displayName': 'Rondale Moore', 'officialPosition': 'WR', 'early_routes': '42', 'late_routes': '55', 'early_mean_soe': '2.215', 'late_mean_soe': '1.716', 'late_minus_early': '-0.498'}, {'nflId': '43010', 'displayName': 'Adam Humphries', 'officialPosition': 'WR', 'early_routes': '92', 'late_routes': '102', 'early_mean_soe': '0.706', 'late_mean_soe': '-0.274', 'late_minus_early': '-0.980'}, {'nflId': '52622', 'displayName': 'Freddie Swain', 'officialPosition': 'WR', 'early_routes': '65', 'late_routes': '82', 'early_mean_soe': '0.704', 'late_mean_soe': '0.377', 'late_minus_early': '-0.327'}, {'nflId': '46256', 'displayName': 'Ray-Ray McCloud', 'officialPosition': 'WR', 'early_routes': '36', 'late_routes': '41', 'early_mean_soe': '0.635', 'late_mean_soe': '-0.304', 'late_minus_early': '-0.939'}, {'nflId': '42448', 'displayName': 'Jamison Crowder', 'officialPosition': 'WR', 'early_routes': '20', 'late_routes': '92', 'early_mean_soe': '0.551', 'late_mean_soe': '0.302', 'l

## Early vs Late Correlation

The basic test is Pearson correlation between early mean SOE and late mean SOE.

In [4]:
early = [float(row["early_mean_soe"]) for row in stability]
late = [float(row["late_mean_soe"]) for row in stability]
correlation = pearson_correlation(early, late)
correlation

0.42018698956496114

In [5]:
print(f"Qualified WRs: {len(stability)}")
print(f"Early mean SOE average: {statistics.fmean(early):.3f}")
print(f"Late mean SOE average: {statistics.fmean(late):.3f}")
print(f"Early/late Pearson correlation: {correlation:.3f}")

Qualified WRs: 107
Early mean SOE average: 0.013
Late mean SOE average: 0.014
Early/late Pearson correlation: 0.420


## Late-Period Leaders

The late-period leaderboard helps identify players whose later SOE supports the early signal and players who surged later.

In [6]:
late_leaders = sorted(stability, key=lambda row: float(row["late_mean_soe"]), reverse=True)
late_leaders[:15]

[{'nflId': '53478', 'displayName': 'Rondale Moore', 'officialPosition': 'WR', 'early_routes': '42', 'late_routes': '55', 'early_mean_soe': '2.215', 'late_mean_soe': '1.716', 'late_minus_early': '-0.498'}, {'nflId': '38613', 'displayName': 'Mohamed Sanu', 'officialPosition': 'WR', 'early_routes': '68', 'late_routes': '44', 'early_mean_soe': '-0.221', 'late_mean_soe': '1.076', 'late_minus_early': '1.297'}, {'nflId': '46095', 'displayName': 'Calvin Ridley', 'officialPosition': 'WR', 'early_routes': '155', 'late_routes': '29', 'early_mean_soe': '0.054', 'late_mean_soe': '0.899', 'late_minus_early': '0.845'}, {'nflId': '42899', 'displayName': 'Damiere Byrd', 'officialPosition': 'WR', 'early_routes': '46', 'late_routes': '26', 'early_mean_soe': '0.078', 'late_mean_soe': '0.859', 'late_minus_early': '0.781'}, {'nflId': '42437', 'displayName': 'Ty Montgomery', 'officialPosition': 'WR', 'early_routes': '33', 'late_routes': '37', 'early_mean_soe': '0.205', 'late_mean_soe': '0.798', 'late_minus_e

## Biggest Changes

Large changes are useful audit cases. Some are real role changes, some are sample noise, and some may expose model limitations.

In [7]:
improvers = sorted(stability, key=lambda row: float(row["late_minus_early"]), reverse=True)
decliners = sorted(stability, key=lambda row: float(row["late_minus_early"]))
improvers[:10]

[{'nflId': '38613', 'displayName': 'Mohamed Sanu', 'officialPosition': 'WR', 'early_routes': '68', 'late_routes': '44', 'early_mean_soe': '-0.221', 'late_mean_soe': '1.076', 'late_minus_early': '1.297'}, {'nflId': '46095', 'displayName': 'Calvin Ridley', 'officialPosition': 'WR', 'early_routes': '155', 'late_routes': '29', 'early_mean_soe': '0.054', 'late_mean_soe': '0.899', 'late_minus_early': '0.845'}, {'nflId': '42899', 'displayName': 'Damiere Byrd', 'officialPosition': 'WR', 'early_routes': '46', 'late_routes': '26', 'early_mean_soe': '0.078', 'late_mean_soe': '0.859', 'late_minus_early': '0.781'}, {'nflId': '52500', 'displayName': 'Devin Duvernay', 'officialPosition': 'WR', 'early_routes': '59', 'late_routes': '70', 'early_mean_soe': '-0.558', 'late_mean_soe': '0.139', 'late_minus_early': '0.697'}, {'nflId': '52536', 'displayName': 'Gabe Davis', 'officialPosition': 'WR', 'early_routes': '55', 'late_routes': '25', 'early_mean_soe': '-0.309', 'late_mean_soe': '0.385', 'late_minus_ea

In [8]:
decliners[:10]

[{'nflId': '43010', 'displayName': 'Adam Humphries', 'officialPosition': 'WR', 'early_routes': '92', 'late_routes': '102', 'early_mean_soe': '0.706', 'late_mean_soe': '-0.274', 'late_minus_early': '-0.980'}, {'nflId': '46256', 'displayName': 'Ray-Ray McCloud', 'officialPosition': 'WR', 'early_routes': '36', 'late_routes': '41', 'early_mean_soe': '0.635', 'late_mean_soe': '-0.304', 'late_minus_early': '-0.939'}, {'nflId': '40026', 'displayName': 'Marquise Goodwin', 'officialPosition': 'WR', 'early_routes': '37', 'late_routes': '50', 'early_mean_soe': '0.473', 'late_mean_soe': '-0.316', 'late_minus_early': '-0.790'}, {'nflId': '45607', 'displayName': 'Greg Ward', 'officialPosition': 'WR', 'early_routes': '32', 'late_routes': '28', 'early_mean_soe': '-0.040', 'late_mean_soe': '-0.580', 'late_minus_early': '-0.539'}, {'nflId': '44896', 'displayName': 'Chris Godwin', 'officialPosition': 'WR', 'early_routes': '158', 'late_routes': '115', 'early_mean_soe': '-0.281', 'late_mean_soe': '-0.799',

## Takeaway

With the current baseline, WR SOE has a split-half correlation around 0.42 among qualifying receivers. That is a promising amount of repeatable signal for a first tracking-data metric, but it is not strong enough to treat the leaderboard as a final player ranking.

The next modeling improvements should try to reduce route/context noise:

1. Add route-shape clusters so verticals, crossers, screens, and quick outs are not over-pooled.
2. Improve defender context beyond nearest defender.
3. Compare stability before and after those improvements.